In [1]:
import os
import re
import numpy as np
import sympy as sp
import openturns as ot
import matplotlib.pyplot as plt
import trimesh as tr

from math import pi
from joblib import Parallel, delayed

from scipy.optimize import minimize, basinhopping, \
                           differential_evolution, brute, shgo, check_grad, \
                           approx_fprime, fsolve, NonlinearConstraint, Bounds, approx_fprime

import otaf

from gldpy import GLD

ot.Log.Show(ot.Log.NONE)
np.set_printoptions(suppress=True)
ar = np.array

# Solving issues regarding the handling of cylinders. 

Cylinders are kinda a pain in the ass to handle, as such here a very simple example where we have two concentric cylinders, one within each other, the outer one fixed, the inner one has translational and rotational play. So the parts are just one feature each.

In [10]:
# Some basic values:
height = 50
RAD1 = 20
RAD2 = 22

R0 = ar([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
x_, y_, z_ = R0[0], R0[1], R0[2]

Frame1 = R0 #For now we can simply align them. Cylinders are oriented such that their axis is on x, the direction doesn't matter.
Frame2 = ar([-x_,y_,-z_]) #If they are the same there is no issue. Let's invert them. 

In [26]:
system_data = {
    "PARTS" : {
        '0' : {
            "a" : {
                "FRAME": Frame1, # Frame doesn't really matter, as long as x is aligned on the axis
                "ORIGIN": ar([0,0,0]), 
                "TYPE": "cylinder",
                "RADIUS": RAD2,
                "EXTENT_LOCAL": {"x_max": height/2, "x_min": -height/2},
                "INTERACTIONS": [f"P1a"], 
                "SURFACE_DIRECTION": "centripetal", #This feature is a hole
                "CONSTRAINTS_D": ["PERFECT"], # No defects, does not move
                "BLOCK_ROTATIONS_G": 'xyz', # Does not rotate 
                "BLOCK_TRANSLATIONS_G": 'xyz', # Does not slide        
            }
        },
        '1' : {
            "a" : {
                "FRAME": Frame2, # Frame doesn't really matter, as long as x is aligned on the axis
                "ORIGIN": ar([2,0,0]), 
                "TYPE": "cylinder",
                "RADIUS": RAD1,
                "EXTENT_LOCAL": {"x_max": height/2, "x_min": -height/2},
                "INTERACTIONS": [f"P0a"], 
                "SURFACE_DIRECTION": "centrifugal", #This feature is a cylinder        
            }
        }  
    },
    "LOOPS": {
        "COMPATIBILITY": {"LO" : "P0aA0 -> P1aA0" },
    },
    "GLOBAL_CONSTRAINTS": "3D",
}

In [27]:
system_data

{'PARTS': {'0': {'a': {'FRAME': array([[1, 0, 0],
           [0, 1, 0],
           [0, 0, 1]]),
    'ORIGIN': array([0, 0, 0]),
    'TYPE': 'cylinder',
    'RADIUS': 22,
    'EXTENT_LOCAL': {'x_max': 25.0, 'x_min': -25.0},
    'INTERACTIONS': ['P1a'],
    'SURFACE_DIRECTION': 'centripetal',
    'CONSTRAINTS_D': ['PERFECT'],
    'BLOCK_ROTATIONS_G': 'xyz',
    'BLOCK_TRANSLATIONS_G': 'xyz'}},
  '1': {'a': {'FRAME': array([[-1,  0,  0],
           [ 0,  1,  0],
           [ 0,  0, -1]]),
    'ORIGIN': array([2, 0, 0]),
    'TYPE': 'cylinder',
    'RADIUS': 20,
    'EXTENT_LOCAL': {'x_max': 25.0, 'x_min': -25.0},
    'INTERACTIONS': ['P0a'],
    'SURFACE_DIRECTION': 'centrifugal'}}},
 'LOOPS': {'COMPATIBILITY': {'LO': 'P0aA0 -> P1aA0'}},
 'GLOBAL_CONSTRAINTS': '3D'}

In [21]:
SDA = otaf.AssemblyDataProcessor(system_data)
SDA.generate_expanded_loops()

In [22]:
CLH = otaf.CompatibilityLoopHandling(SDA)
compatibility_expressions = CLH.get_compatibility_expression_from_FO_matrices()

In [23]:
ILH = otaf.InterfaceLoopHandling(SDA, CLH, circle_resolution=8)
interface_constraints = [expr.evalf(9) for expr in ILH.get_interface_loop_expressions()] 

In [24]:
SOCAM = otaf.SystemOfConstraintsAssemblyModel(
    compatibility_expressions, interface_constraints
)

SOCAM.embedOptimizationVariable()

print(len(SOCAM.deviation_symbols), SOCAM.deviation_symbols)

4 [v_d_1, w_d_1, beta_d_1, gamma_d_1]


In [25]:
SOCAM.test_zero_deviation_feasibility()

{'success': True,
 'status': 0,
 'message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)',
 'gap_values': array([2.]),
 'objective': -2.0}